# Python Statistical Analysis in Practice — 03: Probability Distributions

*Author: Jason JJ Li · Peking University Institute of Population Research*

Statistics is the science of reasoning under uncertainty. Probability distributions
are the mathematical language we use to describe that uncertainty — they tell us
**which values are likely, how spread the data is, and what shape to expect**.

**What you will learn today:**

1. PDF, PMF, CDF: the three faces of a distribution
2. The Normal distribution in depth: z-scores, the 68-95-99.7 rule, quantile functions
3. Discrete distributions: Binomial and Poisson
4. Continuous distributions: Uniform, Exponential, Chi-squared, t, F
5. Fitting a distribution to real (simulated) data with `scipy.stats`
6. Q-Q plots: the visual normality test
7. Kolmogorov-Smirnov & Shapiro-Wilk goodness-of-fit tests
8. How to choose the right distribution: a decision guide
9. The Central Limit Theorem — first look


---

## Part 1: Setup


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import scipy
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)

COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#A23B72',
    'accent':    '#F18F01',
    'success':   '#27AE60',
    'danger':    '#E74C3C',
    'neutral':   '#95A5A6',
    'purple':    '#8E44AD',
}
RNG = np.random.default_rng(42)

print('Libraries loaded.')
print(f'NumPy {np.__version__}  |  Pandas {pd.__version__}  |  SciPy {scipy.__version__}')


---

## Part 2: The Three Faces of a Distribution

Every probability distribution can be described by three equivalent functions:

| Function | Symbol | Answers the question |
|---|---|---|
| **PDF** (Probability Density Function) | $f(x)$ | How dense is the probability at value $x$? |
| **PMF** (Probability Mass Function) | $P(X=x)$ | Discrete equivalent of PDF: what is the exact probability of $x$? |
| **CDF** (Cumulative Distribution Function) | $F(x) = P(X \le x)$ | What fraction of data falls at or below $x$? |

These three are mathematically linked:
$$F(x) = \int_{-\infty}^{x} f(t)\,dt \quad \text{(continuous)}$$
$$F(x) = \sum_{k \le x} P(X=k) \quad \text{(discrete)}$$

### 2.1 PDF and CDF: The Normal Example


In [ ]:
mu, sigma = 100, 15   # IQ score scale
x = np.linspace(40, 160, 500)

pdf_vals = stats.norm.pdf(x, loc=mu, scale=sigma)
cdf_vals = stats.norm.cdf(x, loc=mu, scale=sigma)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Normal Distribution: PDF and CDF', fontsize=13, fontweight='bold')

# PDF
ax1.plot(x, pdf_vals, color=COLORS['primary'], linewidth=2.5, label='PDF')
ax1.fill_between(x, pdf_vals, where=(x >= 85) & (x <= 115),
                 color=COLORS['primary'], alpha=0.25,
                 label='P(85 <= X <= 115)')
ax1.axvline(mu, color=COLORS['danger'], linestyle='--', lw=1.5, label=f'mean={mu}')
ax1.set_xlabel('IQ score')
ax1.set_ylabel('Density')
ax1.set_title('PDF — probability density at each value')
ax1.legend()

# CDF
ax2.plot(x, cdf_vals, color=COLORS['secondary'], linewidth=2.5, label='CDF')
# Annotate specific quantiles
for q, color in [(0.16, COLORS['accent']), (0.50, COLORS['danger']), (0.84, COLORS['success'])]:
    x_q = stats.norm.ppf(q, loc=mu, scale=sigma)
    ax2.annotate(f'P(X<={x_q:.0f})={q:.2f}',
                 xy=(x_q, q), xytext=(x_q - 20, q + 0.08),
                 arrowprops=dict(arrowstyle='->', color='black', lw=1),
                 fontsize=9)
    ax2.scatter([x_q], [q], color=color, s=60, zorder=5)
ax2.axhline(0.5, color=COLORS['danger'], linestyle=':', lw=1)
ax2.set_xlabel('IQ score')
ax2.set_ylabel('Cumulative probability')
ax2.set_title('CDF — fraction of data below x')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Area under PDF from 85 to 115: {stats.norm.cdf(115, mu, sigma) - stats.norm.cdf(85, mu, sigma):.4f}')
print(f'Same, using CDF directly:      {stats.norm.cdf(115, mu, sigma) - stats.norm.cdf(85, mu, sigma):.4f}')


### 2.2 The Quantile (Inverse CDF) Function

The quantile function $Q(p) = F^{-1}(p)$ answers the reverse question:
**"At what value $x$ does the CDF reach $p$?"**

This is how we compute percentiles, critical values, and confidence interval bounds.

In `scipy.stats`, the quantile function is called `.ppf()` (percent-point function).


In [ ]:
print('=== Normal(100, 15) quantile examples ===')
print()
quantiles_of_interest = [0.01, 0.025, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.975, 0.99]

print(f'  {"Percentile":>12}  {"x value":>10}  {"Interpretation"}')
print('  ' + '-'*65)
for p in quantiles_of_interest:
    x_val = stats.norm.ppf(p, loc=mu, scale=sigma)
    print(f'  {p*100:>11.1f}%  {x_val:>10.2f}  P(X <= {x_val:.2f}) = {p:.3f}')

print()
print('Useful special values:')
print(f'  Bottom 5%  cutoff: {stats.norm.ppf(0.05, mu, sigma):.2f}')
print(f'  Top    5%  cutoff: {stats.norm.ppf(0.95, mu, sigma):.2f}')
print(f'  Middle 95% range:  [{stats.norm.ppf(0.025,mu,sigma):.2f}, {stats.norm.ppf(0.975,mu,sigma):.2f}]')
print(f'  z* for 95% CI:     ±{stats.norm.ppf(0.975):.4f}  (the famous "1.96")')


---

## Part 3: The Normal Distribution in Depth

### 3.1 The 68-95-99.7 Rule (Empirical Rule)

For any normal distribution with mean $\mu$ and standard deviation $\sigma$:

| Interval | Probability | Description |
|---|---|---|
| $[\mu - \sigma,\ \mu + \sigma]$ | 68.27% | "Within 1 std" |
| $[\mu - 2\sigma,\ \mu + 2\sigma]$ | 95.45% | "Within 2 std" |
| $[\mu - 3\sigma,\ \mu + 3\sigma]$ | 99.73% | "Within 3 std" |

We verify this by simulation: generate a large normal sample and count.


In [ ]:
mu_h, sigma_h = 170, 8   # human heights in cm
n_large = 100_000
sample = RNG.normal(mu_h, sigma_h, n_large)

print('=== Verification of the 68-95-99.7 rule ===')
print(f'  Distribution: Normal({mu_h}, {sigma_h}),  n={n_large:,}')
print()
for k, label in [(1, '1 sigma'), (2, '2 sigma'), (3, '3 sigma')]:
    lo, hi = mu_h - k*sigma_h, mu_h + k*sigma_h
    observed   = np.mean((sample >= lo) & (sample <= hi))
    theoretical = stats.norm.cdf(hi, mu_h, sigma_h) - stats.norm.cdf(lo, mu_h, sigma_h)
    print(f'  [{lo:.0f}, {hi:.0f}]  Observed={observed:.4f}  Theoretical={theoretical:.4f}  Match={abs(observed-theoretical)<0.005}')

print()
print('The simulation matches theory to 4 decimal places.')


In [ ]:
# Visualise the 3 bands
x = np.linspace(mu_h - 4*sigma_h, mu_h + 4*sigma_h, 500)
y = stats.norm.pdf(x, mu_h, sigma_h)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, y, color=COLORS['primary'], linewidth=2.5, label='PDF', zorder=5)

fills = [
    (3, '#27AE60', '99.7% (within 3σ)'),
    (2, '#F18F01', '95.4% (within 2σ)'),
    (1, '#E74C3C', '68.3% (within 1σ)'),
]
for k, color, label in fills:
    lo, hi = mu_h - k*sigma_h, mu_h + k*sigma_h
    xf = np.linspace(lo, hi, 300)
    ax.fill_between(xf, stats.norm.pdf(xf, mu_h, sigma_h), alpha=0.35, color=color, label=label)

# Annotate sigma lines
for k in [1, 2, 3]:
    for sign in [-1, 1]:
        v = mu_h + sign*k*sigma_h
        ax.axvline(v, color='black', alpha=0.3, lw=0.8, linestyle='--')
        ax.annotate(
            f'{"+" if sign > 0 else "-"}{k}σ={v:.0f}',
            xy=(v, 0.001), ha='center', fontsize=8, color='black', alpha=0.7
        )

ax.axvline(mu_h, color='black', lw=1.5, linestyle='-', label=f'mean={mu_h}')
ax.set_xlabel('Height (cm)')
ax.set_ylabel('Density')
ax.set_title('The 68-95-99.7 Rule — Normal(170, 8)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()


### 3.2 Standardisation and Z-scores

Any normal distribution can be **standardised** to the Standard Normal $N(0,1)$:

$$z = \frac{x - \mu}{\sigma}$$

This allows us to use a single table (or one set of functions) for all normal distributions.


In [ ]:
# Demonstrate standardisation: three different normal distributions
# all reduce to the same Z after standardisation
distributions = [
    (100, 15, 115, COLORS['primary'],   'IQ scores'),
    (170, 8,  178, COLORS['secondary'], 'Heights (cm)'),
    (5000,800, 5800, COLORS['accent'],   'Monthly income (yuan)'),
]

print('Standardisation: x -> z = (x - mu) / sigma')
print()
print(f'  {"Distribution":<22}  {"x":>8}  {"mu":>6}  {"sigma":>6}  {"z=(x-mu)/sigma":>16}  {"P(X<x)":>8}')
print('  ' + '-'*75)
for mu_i, sig_i, x_i, color, name in distributions:
    z = (x_i - mu_i) / sig_i
    p = stats.norm.cdf(z)
    print(f'  {name:<22}  {x_i:>8.0f}  {mu_i:>6.0f}  {sig_i:>6.0f}  {z:>16.4f}  {p:>8.4f}')

print()
print('All three x values correspond to z = 1.0 (one standard deviation above mean).')
print('P(X < x) is identical for all three: 0.8413')

# Visualise: same z-score, different scales
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Standardisation: Different Scales, Same Z-score', fontsize=13, fontweight='bold')

for ax, (mu_i, sig_i, x_i, color, name) in zip(axes, distributions):
    x_range = np.linspace(mu_i - 4*sig_i, mu_i + 4*sig_i, 400)
    y_vals  = stats.norm.pdf(x_range, mu_i, sig_i)
    ax.plot(x_range, y_vals, color=color, linewidth=2.5)
    ax.fill_between(x_range, y_vals, where=(x_range <= x_i),
                    color=color, alpha=0.3, label=f'P(X<={x_i:.0f})=0.84')
    ax.axvline(x_i, color='black', lw=1.5, linestyle='--', label=f'x={x_i:.0f} (z=1.0)')
    ax.set_title(f'{name}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


---

## Part 4: Discrete Distributions

### 4.1 Binomial Distribution

Model for **counting successes** in $n$ independent trials, each with success probability $p$.

$$P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$$

**Parameters:** $n$ (trials), $p$ (success probability)
**Mean:** $np$ · **Variance:** $np(1-p)$

**Typical uses:** number of defective items in a batch, coin flips, survey "yes/no" responses,
number of patients who respond to a treatment.


In [ ]:
# Compare Binomial for different p values
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Binomial Distribution B(n=20, p) — Effect of p', fontsize=13, fontweight='bold')

n_trials = 20
p_values = [0.2, 0.5, 0.8]

for ax, p in zip(axes, p_values):
    k_vals = np.arange(0, n_trials + 1)
    pmf    = stats.binom.pmf(k_vals, n=n_trials, p=p)
    cdf    = stats.binom.cdf(k_vals, n=n_trials, p=p)

    bars = ax.bar(k_vals, pmf, color=COLORS['primary'], alpha=0.7, edgecolor='white')
    ax2  = ax.twinx()
    ax2.step(k_vals, cdf, color=COLORS['danger'], linewidth=2.5, where='post', label='CDF')
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel('CDF', color=COLORS['danger'])

    ax.axvline(n_trials * p, color='black', lw=2, linestyle='--',
               label=f'mean=np={n_trials*p:.0f}')
    ax.set_title(f'B(n={n_trials}, p={p})  mean={n_trials*p:.1f}')
    ax.set_xlabel('k (number of successes)')
    ax.set_ylabel('P(X=k)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Verify mean and variance
print('Verification: simulated vs theoretical moments')
print()
for p in p_values:
    sample = RNG.binomial(n=n_trials, p=p, size=50000)
    print(f'  B(n={n_trials}, p={p}):  '
          f'E[X] theory={n_trials*p:.2f} sim={sample.mean():.2f}  |  '
          f'Var theory={n_trials*p*(1-p):.2f} sim={sample.var(ddof=1):.2f}')


### 4.2 Poisson Distribution

Model for **counting rare events** in a fixed interval of time or space.

$$P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad k = 0, 1, 2, \ldots$$

**Parameter:** $\lambda$ (rate = expected count per interval)
**Mean = Variance = $\lambda$** (this equality is a diagnostic: if sample mean ≠ variance, Poisson doesn't fit)

**Typical uses:** hospital admissions per hour, emails per day, mutations per gene, accidents per year.

**Important relationship:** $\text{Binomial}(n, p) \to \text{Poisson}(\lambda=np)$ as $n \to \infty$, $p \to 0$.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Poisson Distribution Pois(lambda): Effect of Rate', fontsize=13, fontweight='bold')

lambdas = [1, 5, 15]

for ax, lam in zip(axes, lambdas):
    k_max  = max(20, int(lam + 4*np.sqrt(lam) + 2))
    k_vals = np.arange(0, k_max + 1)
    pmf    = stats.poisson.pmf(k_vals, mu=lam)
    cdf    = stats.poisson.cdf(k_vals, mu=lam)

    ax.bar(k_vals, pmf, color=COLORS['secondary'], alpha=0.7, edgecolor='white')
    ax2 = ax.twinx()
    ax2.step(k_vals, cdf, color=COLORS['danger'], lw=2.5, where='post', label='CDF')
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel('CDF', color=COLORS['danger'])

    ax.axvline(lam, color='black', lw=2, linestyle='--', label=f'lambda={lam} (mean=var)')
    ax.set_title(f'Poisson(lambda={lam})')
    ax.set_xlabel('k (count)')
    ax.set_ylabel('P(X=k)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Mean = Variance diagnostic:')
for lam in lambdas:
    s = RNG.poisson(lam=lam, size=50000)
    print(f'  Poisson(lambda={lam:2d}):  mean={s.mean():.3f}  variance={s.var(ddof=1):.3f}  ratio={s.mean()/s.var(ddof=1):.3f}')


### 4.3 Binomial → Poisson Approximation

When $n$ is large and $p$ is small (rare events), $B(n, p) \approx \text{Poisson}(np)$.
The approximation works well when $n > 20$ and $p < 0.05$.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Binomial to Poisson Approximation (lambda=np=3)', fontsize=13, fontweight='bold')

lam_fixed = 3  # keep lambda = np = 3 constant
scenarios = [(10, 0.3), (50, 0.06), (300, 0.01)]

for ax, (n, p) in zip(axes, scenarios):
    k_max  = 15
    k_vals = np.arange(0, k_max + 1)
    binom_pmf   = stats.binom.pmf(k_vals, n=n, p=p)
    poisson_pmf = stats.poisson.pmf(k_vals, mu=lam_fixed)

    width = 0.4
    ax.bar(k_vals - width/2, binom_pmf, width=width, color=COLORS['primary'],
           alpha=0.8, edgecolor='white', label=f'Binom(n={n}, p={p})')
    ax.bar(k_vals + width/2, poisson_pmf, width=width, color=COLORS['accent'],
           alpha=0.8, edgecolor='white', label='Poisson(3)')
    ax.set_title(f'n={n}, p={p}  (np={lam_fixed})')
    ax.set_xlabel('k')
    ax.set_ylabel('P(X=k)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('As n increases and p decreases (with np constant), the Binomial converges to Poisson.')


---

## Part 5: Continuous Distributions Beyond the Normal

### 5.1 Uniform Distribution

$X \sim \text{Uniform}(a, b)$ — every value in $[a, b]$ is equally likely.

$$f(x) = \frac{1}{b-a}, \quad a \le x \le b$$

**Mean:** $\frac{a+b}{2}$ · **Variance:** $\frac{(b-a)^2}{12}$

**Uses:** Random number generation, initial values in algorithms, discrete die rolls.


### 5.2 Exponential Distribution

$X \sim \text{Exp}(\lambda)$ — time until the **first** occurrence of a Poisson event.

$$f(x) = \lambda e^{-\lambda x}, \quad x \ge 0$$

**Mean:** $1/\lambda$ · **Variance:** $1/\lambda^2$

**Memoryless property:** $P(X > s+t \mid X > s) = P(X > t)$ — past history doesn't matter.

**Uses:** time between arrivals, machine failure times, survival analysis (simple model).


In [ ]:
# Side-by-side: Uniform, Exponential, and their CDF shapes
fig = plt.figure(figsize=(16, 8))
fig.suptitle('Continuous Distributions: PDF and CDF Gallery', fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.35)

dist_configs = [
    ('Uniform(0,1)',        stats.uniform(0,1),       np.linspace(-0.1,1.1,400),  COLORS['primary']),
    ('Exponential(rate=1)', stats.expon(scale=1),     np.linspace(-0.1,5,400),    COLORS['secondary']),
    ('Chi-squared(df=5)',   stats.chi2(df=5),          np.linspace(0,20,400),      COLORS['accent']),
    ('t(df=5)',             stats.t(df=5),             np.linspace(-5,5,400),      COLORS['success']),
]

for col, (name, dist, x_range, color) in enumerate(dist_configs):
    ax_pdf = fig.add_subplot(gs[0, col])
    ax_cdf = fig.add_subplot(gs[1, col])

    pdf_v = dist.pdf(x_range)
    cdf_v = dist.cdf(x_range)

    ax_pdf.plot(x_range, pdf_v, color=color, lw=2.5)
    ax_pdf.fill_between(x_range, pdf_v, alpha=0.2, color=color)
    ax_pdf.set_title(f'{name}')
    ax_pdf.set_ylabel('Density')
    ax_pdf.set_xlabel('x')

    ax_cdf.plot(x_range, cdf_v, color=color, lw=2.5)
    ax_cdf.set_title('CDF')
    ax_cdf.set_ylabel('P(X <= x)')
    ax_cdf.set_xlabel('x')
    ax_cdf.axhline(0.5, color='black', lw=1, linestyle=':', alpha=0.5)

plt.show()


### 5.3 The t-Distribution

The t-distribution looks like a Normal but has **heavier tails** (controlled by degrees of freedom $\nu$):

$$f(t) \propto \left(1 + \frac{t^2}{\nu}\right)^{-(\nu+1)/2}$$

- As $\nu \to \infty$: t-distribution → Standard Normal
- For small $\nu$ (e.g. 3-5): much heavier tails, extreme values much more common
- **When we use it:** confidence intervals and hypothesis tests for the mean when $\sigma$ is unknown
  (which is always in practice)


In [ ]:
x = np.linspace(-5, 5, 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('t-Distribution: Heavier Tails than Normal', fontsize=13, fontweight='bold')

dfs       = [1, 3, 5, 10, 30]
t_colors  = [COLORS['danger'], COLORS['accent'], COLORS['primary'], COLORS['success'], COLORS['purple']]

for df, color in zip(dfs, t_colors):
    pdf_t = stats.t.pdf(x, df=df)
    ax1.plot(x, pdf_t, color=color, lw=2, label=f'df={df}')

ax1.plot(x, stats.norm.pdf(x), color='black', lw=2.5, linestyle='--', label='Normal')
ax1.set_xlim(-5, 5)
ax1.set_xlabel('t')
ax1.set_ylabel('Density')
ax1.set_title('PDF: t vs Normal')
ax1.legend(fontsize=9)

# Zoom into tails
ax2.plot(x, stats.norm.pdf(x), color='black', lw=2.5, linestyle='--', label='Normal')
for df, color in zip([3, 5, 10], [COLORS['danger'], COLORS['accent'], COLORS['primary']]):
    ax2.plot(x, stats.t.pdf(x, df=df), color=color, lw=2, label=f'df={df}')
ax2.set_xlim(2, 5)
ax2.set_ylim(0, 0.03)
ax2.set_xlabel('|t|')
ax2.set_ylabel('Density (zoom into right tail)')
ax2.set_title('Right-tail zoom: t has much more probability mass')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Probability P(|X| > 2):')
print(f'  Normal:    {2*stats.norm.sf(2):.6f}')
for df in [3, 5, 10, 30]:
    print(f'  t(df={df:2d}): {2*stats.t.sf(2, df=df):.6f}  ({2*stats.t.sf(2, df=df)/2*stats.norm.sf(2)*100:.0f}x more than Normal)' if df > 0 else '')


---

## Part 6: Fitting a Distribution to Data

Given a dataset, `scipy.stats` can **estimate the best-fit parameters** for
any named distribution using Maximum Likelihood Estimation (MLE).

### 6.1 Fitting the Normal Distribution


In [ ]:
# Simulate data and fit back
TRUE_MU, TRUE_SIGMA = 170, 8
sample_heights = RNG.normal(TRUE_MU, TRUE_SIGMA, size=500)

# MLE fit
fit_mu, fit_sigma = stats.norm.fit(sample_heights)
print('=== Fitting Normal to simulated height data (n=500) ===')
print(f'  True   : mu={TRUE_MU},  sigma={TRUE_SIGMA}')
print(f'  MLE fit: mu={fit_mu:.3f},  sigma={fit_sigma:.3f}')
print(f'  Bias   : mu_bias={fit_mu-TRUE_MU:+.3f},  sigma_bias={fit_sigma-TRUE_SIGMA:+.3f}')

# Visualise fit
x_range = np.linspace(140, 200, 400)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sample_heights, bins=35, density=True, color=COLORS['primary'],
        edgecolor='white', alpha=0.7, label='Simulated data')
ax.plot(x_range, stats.norm.pdf(x_range, TRUE_MU, TRUE_SIGMA),
        color=COLORS['danger'], lw=2.5, linestyle='-', label=f'True: N({TRUE_MU}, {TRUE_SIGMA})')
ax.plot(x_range, stats.norm.pdf(x_range, fit_mu, fit_sigma),
        color=COLORS['success'], lw=2, linestyle='--',
        label=f'MLE fit: N({fit_mu:.1f}, {fit_sigma:.1f})')
ax.set_xlabel('Height (cm)')
ax.set_ylabel('Density')
ax.set_title('MLE Fitting: True Distribution vs Fitted Distribution')
ax.legend()
plt.tight_layout()
plt.show()


### 6.2 Fitting Multiple Distributions — Which One Fits?

When we don't know the true distribution, we can try several candidates and compare.
We use Kolmogorov-Smirnov (KS) test to measure goodness of fit.


In [ ]:
# Generate log-normal data (mimicking right-skewed income)
TRUE_LNMU, TRUE_LNSIG = 10.5, 0.6
income_sample = RNG.lognormal(TRUE_LNMU, TRUE_LNSIG, size=1000)

# Candidates to fit
candidates = {
    'Normal':      stats.norm,
    'Log-normal':  stats.lognorm,
    'Exponential': stats.expon,
    'Gamma':       stats.gamma,
}

print('=== Fitting multiple distributions to right-skewed income data ===')
print()
print(f'  {"Distribution":<14}  {"KS statistic":>13}  {"p-value":>10}  {"Likely fit?"}')
print('  ' + '-'*60)

fit_results = {}
for name, dist in candidates.items():
    params = dist.fit(income_sample)
    ks_stat, ks_p = stats.kstest(income_sample, dist.cdf, args=params)
    good = 'YES' if ks_p > 0.05 else 'no'
    print(f'  {name:<14}  {ks_stat:>13.6f}  {ks_p:>10.4f}  {good}')
    fit_results[name] = (params, ks_stat, ks_p)

print()
print('KS test: H0 = data comes from this distribution')
print('  p > 0.05  -> cannot reject H0  -> distribution is plausible')
print('  p < 0.05  -> reject H0         -> distribution does not fit well')


In [ ]:
# Visualise all fits
x_range = np.linspace(income_sample.min(), np.percentile(income_sample, 99), 400)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparing Distribution Fits to Income Data', fontsize=12, fontweight='bold')

# Left: PDF comparison
axes[0].hist(income_sample, bins=60, density=True, color=COLORS['neutral'],
             edgecolor='white', alpha=0.7, label='Income data')

fit_colors = [COLORS['danger'], COLORS['success'], COLORS['accent'], COLORS['secondary']]
for (name, dist), color in zip(candidates.items(), fit_colors):
    params_fit, ks, pval = fit_results[name]
    fitted_pdf = dist.pdf(x_range, *params_fit)
    lw = 3 if pval > 0.05 else 1.5
    ls = '-' if pval > 0.05 else '--'
    axes[0].plot(x_range, fitted_pdf, color=color, lw=lw, linestyle=ls,
                 label=f'{name} (p={pval:.3f})')

axes[0].set_xlim(x_range[0], x_range[-1])
axes[0].set_xlabel('Income')
axes[0].set_ylabel('Density')
axes[0].set_title('PDF fits (solid line = good fit p>0.05)')
axes[0].legend(fontsize=8)

# Right: log-scale x-axis to see shape better
axes[1].hist(np.log(income_sample), bins=50, density=True,
             color=COLORS['primary'], edgecolor='white', alpha=0.7,
             label='log(Income)')
# After log-transform, log-normal becomes normal
ln_mu_fit, ln_sigma_fit = np.mean(np.log(income_sample)), np.std(np.log(income_sample), ddof=1)
x_log = np.linspace(np.log(income_sample.min()), np.log(np.percentile(income_sample,99)), 400)
axes[1].plot(x_log, stats.norm.pdf(x_log, ln_mu_fit, ln_sigma_fit),
             color=COLORS['danger'], lw=3, label='Normal fit to log(Income)')
axes[1].set_xlabel('log(Income)')
axes[1].set_ylabel('Density')
axes[1].set_title('Log-transformed data: log-normal -> normal')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
print('After log-transforming, the data looks Normal -> confirms Log-normal is the right model.')


---

## Part 7: Q-Q Plots — The Visual Normality Test

A Quantile-Quantile (Q-Q) plot compares the **empirical quantiles** of your data
against the **theoretical quantiles** of a reference distribution.

### How to read a Q-Q plot:
- **Points on the diagonal line** → data matches the reference distribution
- **S-curve** → heavier tails than reference (kurtosis)
- **Curve bending up** at the right → right skewness
- **Outliers at corners** → extreme values beyond what the distribution predicts


In [ ]:
# Four datasets with different shapes
datasets = {
    'Normal (should fit)':    RNG.normal(0, 1, 300),
    'Right-skewed':           RNG.lognormal(0, 0.7, 300),
    'Left-skewed':            -RNG.lognormal(0, 0.7, 300),
    'Heavy-tailed (t, df=3)': RNG.standard_t(df=3, size=300),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Q-Q Plots: Recognising Departure from Normality', fontsize=13, fontweight='bold')

for col, (name, data) in enumerate(datasets.items()):
    # Top row: histogram
    ax_hist = axes[0, col]
    ax_hist.hist(data, bins=40, color=COLORS['primary'], edgecolor='white',
                 alpha=0.8, density=True)
    x_r = np.linspace(data.min(), data.max(), 200)
    ax_hist.plot(x_r, stats.norm.pdf(x_r, data.mean(), data.std(ddof=1)),
                 color=COLORS['danger'], lw=2, label='Normal')
    ax_hist.set_title(f'{name}')
    ax_hist.set_xlabel('Value')
    ax_hist.legend(fontsize=7)

    # Bottom row: Q-Q plot
    ax_qq = axes[1, col]
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm', fit=True)
    ax_qq.scatter(osm, osr, s=10, alpha=0.5, color=COLORS['primary'])
    # Reference line
    x_line = np.array([min(osm), max(osm)])
    ax_qq.plot(x_line, slope * x_line + intercept,
               color=COLORS['danger'], lw=2, label=f'R={r:.3f}')
    ax_qq.set_xlabel('Theoretical quantiles')
    ax_qq.set_ylabel('Sample quantiles')
    ax_qq.set_title('Q-Q plot')
    ax_qq.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Reading guide:')
print('  Normal       : points follow the line closely -> confirms normality')
print('  Right-skewed : upper end curves UP above the line -> right heavy tail')
print('  Left-skewed  : lower end curves DOWN below the line -> left heavy tail')
print('  Heavy-tailed : both ends bend away from the line -> fat tails (S-curve)')


---

## Part 8: Formal Goodness-of-Fit Tests

Visual inspection (histogram, Q-Q plot) should always come first. But formal tests
give you a **p-value** for the "is this distribution plausible?" question.

### Two main tests for normality:

| Test | Null hypothesis | Best used for |
|---|---|---|
| **Shapiro-Wilk** | Data is normal | Small to medium samples (n < 5000) — most powerful |
| **Kolmogorov-Smirnov** | Data matches specified distribution | Any distribution; less powerful than S-W |
| **Anderson-Darling** | Data matches specified distribution | Better than KS for tails |

> **Important caveat:** with large n, these tests will reject normality for tiny,
> practically irrelevant deviations. Always look at the Q-Q plot alongside the p-value.


In [ ]:
# Demonstrate: Shapiro-Wilk on normal vs non-normal data, various n
print('=== Shapiro-Wilk Normality Test ===')
print()
print(f'  {"Dataset":<28}  {"n":>5}  {"W statistic":>12}  {"p-value":>10}  {"Reject H0?"}')
print('  ' + '-'*75)

test_cases = [
    ('Normal(0,1)',              lambda n: RNG.normal(0, 1, n)),
    ('t(df=5) — slightly fat',  lambda n: RNG.standard_t(df=5, size=n)),
    ('Log-normal — skewed',     lambda n: RNG.lognormal(0, 0.5, n)),
    ('Uniform(0,1)',             lambda n: RNG.uniform(0, 1, n)),
]

for n in [50, 200, 1000]:
    if n > 50:
        print()
    for name, gen in test_cases:
        data = gen(n)
        w, p = stats.shapiro(data)
        reject = 'YES  <--' if p < 0.05 else 'no'
        print(f'  {name:<28}  {n:>5}  {w:>12.6f}  {p:>10.4f}  {reject}')


In [ ]:
# The large-n caveat: tiny deviation from normality gets flagged
print('=== Large-n caveat: practical vs statistical significance ===')
print()
print('  Generating Normal(0,1) data with a tiny skewness perturbation (0.02)...')
print()

for n in [100, 1000, 5000, 20000, 100000]:
    # Add a tiny perturbation: slightly skew the data
    base = RNG.normal(0, 1, n)
    perturbed = base + 0.02 * base**2  # tiny non-linearity
    w, p = stats.shapiro(min(perturbed, key=None) if n > 5000 else perturbed[:5000])
    w2, p2 = stats.shapiro(perturbed[:min(n, 5000)])  # SW limited to 5000
    _, ks_p = stats.kstest(perturbed, 'norm', args=(perturbed.mean(), perturbed.std(ddof=1)))
    skew_val = stats.skew(perturbed)
    print(f'  n={n:>7,}  skewness={skew_val:.4f}  SW: p={p2:.4f}  KS: p={ks_p:.6f}  '
          f'SW_reject={"YES" if p2<0.05 else "no"}')

print()
print('Lesson: At n=100,000, even a skewness of 0.04 yields p<0.05.')
print('Always ask: "Is this deviation practically meaningful?" not just "Is p<0.05?"')


---

## Part 9: Choosing the Right Distribution — A Decision Guide

| Your variable | Typical distribution(s) | Key check |
|---|---|---|
| Continuous, symmetric, no bounds | **Normal** | Q-Q plot, Shapiro-Wilk |
| Continuous, strictly positive, right-skewed | **Log-normal** or **Gamma** | Log-transform -> Normal? |
| Continuous, bounded [0,1] (proportions) | **Beta** | values clamped to [0,1]? |
| Continuous, time until event | **Exponential** or **Weibull** | Memoryless? |
| Count of events in interval | **Poisson** | Mean ≈ Variance? |
| Count of successes in n trials | **Binomial** | Fixed n, binary outcome? |
| Count with overdispersion (var >> mean) | **Negative Binomial** | Variance >> Mean? |
| Ordinal or bounded integer | **Ordered Logit** / **Multinomial** | Ordinal scale? |


In [ ]:
# Decision tree simulation: generate data from 4 different processes,
# then use diagnostics to identify the right distribution

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle('Distribution Identification Toolkit', fontsize=13, fontweight='bold')

scenarios = [
    ('Hospital stay days',      RNG.exponential(scale=4, size=500),     stats.expon,   'Exp'),
    ('Log-normal income',       RNG.lognormal(10, 0.5, size=500),       stats.lognorm, 'LogN'),
    ('Count: arrivals/hour',    RNG.poisson(lam=8, size=500).astype(float), stats.poisson, 'Pois'),
    ('Normal height',           RNG.normal(170, 8, size=500),           stats.norm,    'Normal'),
]

for col, (name, data, true_dist, dist_label) in enumerate(scenarios):
    # Row 0: histogram
    ax_h = axes[0, col]
    ax_h.hist(data, bins=35, density=True, color=COLORS['primary'],
              edgecolor='white', alpha=0.75)
    ax_h.set_title(f'{name}')
    ax_h.set_xlabel('Value')
    ax_h.set_ylabel('Density')

    # Summary diagnostics
    skew_v = stats.skew(data)
    kurt_v = stats.kurtosis(data)
    ax_h.text(0.97, 0.97, f'skew={skew_v:.2f}',
              transform=ax_h.transAxes, ha='right', va='top', fontsize=8,
              bbox=dict(boxstyle='round', fc='white', alpha=0.7))

    # Row 1: Q-Q plot against Normal
    ax_q = axes[1, col]
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm', fit=True)
    ax_q.scatter(osm, osr, s=8, alpha=0.5, color=COLORS['primary'])
    x_qq = np.array([osm.min(), osm.max()])
    ax_q.plot(x_qq, slope*x_qq + intercept, color=COLORS['danger'], lw=2)
    ax_q.set_title(f'Q-Q vs Normal (R={r:.3f})')
    ax_q.set_xlabel('Theoretical')
    ax_q.set_ylabel('Sample')

plt.tight_layout()
plt.show()

print('Diagnostics summary:')
for name, data, _, label in scenarios:
    sw_stat, sw_p = stats.shapiro(data[:min(500, len(data))])
    skew_v        = stats.skew(data)
    mv_ratio      = data.mean() / data.var(ddof=1) if data.var(ddof=1) > 0 else np.nan
    print(f'  {name:<25}  skew={skew_v:+.2f}  SW_p={sw_p:.4f}  mean/var={mv_ratio:.3f}  -> True dist: {label}')


---

## Part 10: First Look at the Central Limit Theorem

The Central Limit Theorem (CLT) is perhaps the most powerful result in statistics:

> **CLT:** As $n \to \infty$, the sample mean $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$
> converges in distribution to $N(\mu, \sigma^2/n)$, **regardless of the shape of the
> population distribution**.

This is why normal-based methods (t-tests, confidence intervals) work even when
the underlying data is not perfectly normal — as long as $n$ is large enough.

### Rule of thumb: $n \ge 30$ is usually sufficient for CLT to kick in for moderately skewed data.

We will explore CLT in depth in Lecture 04. Here we preview the key simulation.


In [ ]:
# Demonstrate CLT: sample means from highly non-normal populations
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle('Central Limit Theorem Preview: Sample Means Always Become Normal', fontsize=12, fontweight='bold')

populations = [
    ('Uniform(0,1)',      lambda n: RNG.uniform(0, 1, n)),
    ('Exponential(1)',    lambda n: RNG.exponential(1, n)),
    ('Bernoulli(p=0.2)',  lambda n: RNG.binomial(1, 0.2, n).astype(float)),
    ('Bimodal',           lambda n: np.where(RNG.uniform(0,1,n)<0.5, RNG.normal(-2,0.5,n), RNG.normal(2,0.5,n))),
]
N_REPS  = 3000
N_SAMPLE = 50

for col, (name, gen) in enumerate(populations):
    # Row 0: population distribution
    pop = gen(5000)
    axes[0, col].hist(pop, bins=50, density=True, color=COLORS['secondary'],
                      edgecolor='white', alpha=0.75)
    axes[0, col].set_title(f'Population: {name}')
    axes[0, col].set_xlabel('Value')
    axes[0, col].set_ylabel('Density')
    skew_v = stats.skew(pop)
    axes[0, col].text(0.97, 0.97, f'skew={skew_v:.2f}',
                      transform=axes[0, col].transAxes, ha='right', va='top', fontsize=8,
                      bbox=dict(boxstyle='round', fc='white', alpha=0.7))

    # Row 1: sampling distribution of the mean (n=50)
    sample_means = [gen(N_SAMPLE).mean() for _ in range(N_REPS)]
    sm = np.array(sample_means)
    axes[1, col].hist(sm, bins=50, density=True, color=COLORS['primary'],
                      edgecolor='white', alpha=0.75)
    # Overlay expected normal distribution
    expected_mu    = pop.mean()
    expected_sigma = pop.std(ddof=1) / np.sqrt(N_SAMPLE)
    x_range = np.linspace(sm.min(), sm.max(), 200)
    axes[1, col].plot(x_range, stats.norm.pdf(x_range, expected_mu, expected_sigma),
                      color=COLORS['danger'], lw=2.5, label=f'N({expected_mu:.2f}, {expected_sigma:.3f})')
    axes[1, col].set_title(f'Sample means (n={N_SAMPLE}, reps={N_REPS:,})')
    axes[1, col].set_xlabel('Sample mean')
    axes[1, col].set_ylabel('Density')
    axes[1, col].legend(fontsize=7)
    sw_stat, sw_p = stats.shapiro(sm[:500])
    axes[1, col].text(0.97, 0.05, f'SW p={sw_p:.3f}',
                      transform=axes[1, col].transAxes, ha='right', va='bottom',
                      fontsize=8, bbox=dict(boxstyle='round', fc='white', alpha=0.7))

plt.tight_layout()
plt.show()

print('Even when the population is highly non-normal, sample means (n=50) are well-approximated')
print('by a Normal distribution — this is the Central Limit Theorem in action.')


---

## Summary

| Concept | Key takeaway |
|---|---|
| PDF, CDF, quantile | Three equivalent views of any distribution |
| 68-95-99.7 rule | Verified by simulation; true for all Normal$(\mu, \sigma)$ |
| z-score | Standardises any normal to $N(0,1)$; unlocks shared tables |
| Binomial | Count of successes; $\to$ Poisson when $n$ large, $p$ small |
| Poisson | Count of rare events; mean = variance is a diagnostic |
| t-distribution | Heavier tails than Normal; $\to$ Normal as df $\to \infty$ |
| Distribution fitting | `scipy.stats.dist.fit()` + KS test for goodness of fit |
| Q-Q plot | Visual normality test; S-curve = fat tails; bend = skew |
| Shapiro-Wilk | Most powerful normality test; inflated at large $n$ |
| CLT preview | Sample means always become Normal as $n$ grows |

### Distribution Identification Checklist

```
1. Is the variable discrete (counts) or continuous?
2. Continuous:
   a. Is it bounded below at 0? → Log-normal, Gamma, Exponential, Weibull
   b. Is it bounded in [0,1]? → Beta
   c. Symmetric, unbounded? → Normal (check Q-Q plot)
3. Discrete:
   a. Count of events per interval? → Poisson (check mean ≈ variance)
   b. Count of successes in n trials? → Binomial
   c. Count with overdispersion? → Negative Binomial
```

**Next lecture →** Lecture 04: The Central Limit Theorem in Action —
why it works, when it fails, and how it justifies many statistical tests.

---
*Python Statistical Analysis in Practice — Lecture 03*  
*Author: Jason JJ Li · PKU Institute of Population Research*
